# Lightspeed API Explorer

Interactive notebook for exploring the pyLightspeed library across all series.

| Series | Auth | Main endpoint used |
|---|---|---|
| **R-Series** | OAuth token file | `Items` |
| **C-Series** | Basic auth | `Products` |
| **X-Series** | Personal token | `Products` |

Credentials are loaded from the repo's `.env` file — fill that in before running.

## 1. Install & Import Required Libraries

In [5]:

# Install dependencies into the active kernel.
# Also explicitly add the src/ directory to sys.path so pylightspeed is
# importable even if pip's editable-install .pth file isn't loaded yet.
import subprocess, sys
from pathlib import Path

REPO_ROOT = Path("C:/Data/Development/pyLightspeed")
SRC_DIR   = str(REPO_ROOT / "src")

if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "--quiet",
     "python-dotenv", "pandas",
     ]
)
print("Dependencies ready.")
print(f"pylightspeed src on path: {SRC_DIR}")


Dependencies ready.
pylightspeed src on path: C:\Data\Development\pyLightspeed\src


In [6]:
import os, json, pprint
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from pylightspeed.api import (
    LightspeedRSeriesApi,
    LightspeedCSeriesApi,
    LightspeedXSeriesApi,
)

# Load .env from repo root (one level above this notebook)
load_dotenv(Path(".env"))

print("pylightspeed imported successfully.")

2026-02-28 14:42:50.200 | DEBUG    | pylightspeed.connection:<module>:45 - connection module loaded


pylightspeed imported successfully.


## 2. Configure API Connections

Each connection reads credentials from `.env`.  
You can also override them directly in the cell below — just replace the `os.getenv(...)` calls.

> **R-Series** uses an OAuth token file on disk.  
> **C-Series** uses basic auth (API key + secret).  
> **X-Series** uses a Personal Access Token (Plus plan required).

Set any API you don't have credentials for to `None` — the later cells will skip it gracefully.

In [7]:
# ── R-Series (Lightspeed Retail) ─────────────────────────────────────────────
RSR_ACCOUNT_ID  = os.getenv("LSR_ACCOUNT_ID")
RSR_CLIENT_ID   = os.getenv("LSR_CLIENT_ID")
RSR_CLIENT_SECRET = os.getenv("LSR_CLIENT_SECRET")
RSR_TOKEN_FILE  = os.getenv("LSR_TOKEN_FILE")

# ── C-Series (Lightspeed eCom) ────────────────────────────────────────────────
LSC_API_KEY     = os.getenv("LSC_API_KEY")
LSC_API_SECRET  = os.getenv("LSC_API_SECRET")
LSC_API_HOST    = os.getenv("LSC_API_HOST", "api.shoplightspeed.com")
LSC_API_PATH    = os.getenv("LSC_API_PATH", "/us/{}")

# ── X-Series (Lightspeed Retail X) ───────────────────────────────────────────
LSX_DOMAIN_PREFIX  = os.getenv("LSX_DOMAIN_PREFIX")
LSX_PERSONAL_TOKEN = os.getenv("LSX_PERSONAL_TOKEN")

print("Credentials loaded.")
print(f"  R-Series account: {RSR_ACCOUNT_ID  or '⚠ not set'}")
print(f"  C-Series key:     {'set' if LSC_API_KEY else '⚠ not set'}")
print(f"  X-Series domain:  {LSX_DOMAIN_PREFIX or '⚠ not set'}")

Credentials loaded.
  R-Series account: 190211
  C-Series key:     set
  X-Series domain:  ⚠ not set


In [8]:
# Build API objects.  Each one is set to None if credentials are missing
# so the rest of the notebook can check with  `if lsr:` before calling.

lsr = lsc = lsx = None

if all([RSR_ACCOUNT_ID, RSR_CLIENT_ID, RSR_CLIENT_SECRET, RSR_TOKEN_FILE]):
    lsr = LightspeedRSeriesApi(
        account_id=RSR_ACCOUNT_ID,
        client_id=RSR_CLIENT_ID,
        client_secret=RSR_CLIENT_SECRET,
        token_file=RSR_TOKEN_FILE,
    )
    print("✓ R-Series connected")
else:
    print("✗ R-Series skipped (missing credentials)")

if LSC_API_KEY and LSC_API_SECRET:
    lsc = LightspeedCSeriesApi(
        api_key=LSC_API_KEY,
        api_secret=LSC_API_SECRET,
        host=LSC_API_HOST,
        api_path=LSC_API_PATH,
    )
    print("✓ C-Series connected")
else:
    print("✗ C-Series skipped (missing credentials)")

if LSX_DOMAIN_PREFIX and LSX_PERSONAL_TOKEN:
    lsx = LightspeedXSeriesApi(
        domain_prefix=LSX_DOMAIN_PREFIX,
        personal_token=LSX_PERSONAL_TOKEN,
    )
    print("✓ X-Series connected")
else:
    print("✗ X-Series skipped (missing credentials)")

HTTPError: 400 Client Error: Bad Request for url: https://cloud.lightspeedapp.com/auth/oauth/token

## 3. Get a List of Items / Products

`page()` returns the first page of results (up to 100 for R-Series, 50 for C/X-Series by default).  
Pass `limit=N` to control page size; use `listall()` to fetch every page automatically.

### R-Series — Items

In [ ]:
if lsr:
    # Fetch first page (default limit = 100 for R-Series)
    rsr_items = lsr.Items.page(limit=10)
    print(f"Returned {len(rsr_items)} items  |  total in store: {lsr.connection.count}")
    print()

    # Build a tidy summary table from the scalar fields
    rows = [item.as_dict() for item in rsr_items]
    df_rsr = pd.DataFrame(rows)

    # Show a curated subset of columns if they exist
    show_cols = [c for c in ["itemID", "description", "sku", "createTime", "timeStamp"]
                 if c in df_rsr.columns]
    display(df_rsr[show_cols].head(10))
else:
    print("R-Series not configured — skipping.")

### C-Series — Products

In [ ]:
if lsc:
    # C-Series default page size is 50; max is 250
    lsc_products = lsc.Products.page(limit=10)
    print(f"Returned {len(lsc_products)} products")
    print()

    rows = [prod.as_dict() for prod in lsc_products]
    df_lsc = pd.DataFrame(rows)

    show_cols = [c for c in ["id", "title", "sku", "price", "isVisible", "createdAt", "updatedAt"]
                 if c in df_lsc.columns]
    display(df_lsc[show_cols].head(10))
else:
    print("C-Series not configured — skipping.")

### X-Series — Products

In [ ]:
if lsx:
    lsx_products = lsx.Products.page()
    print(f"Returned {len(lsx_products)} products")
    print()

    rows = [prod.as_dict() for prod in lsx_products]
    df_lsx = pd.DataFrame(rows)

    show_cols = [c for c in ["id", "name", "sku", "base_price", "retail_price",
                              "created_at", "updated_at"]
                 if c in df_lsx.columns]
    display(df_lsx[show_cols].head(10))
else:
    print("X-Series not configured — skipping.")

## 4. Get a Single Item

`get(id)` fetches one record by its primary key and returns a full object with dot-access to every field.

**Before running the update cells below**, set `RSR_ITEM_ID`, `LSC_PRODUCT_ID`, and/or `LSX_PRODUCT_ID` to a real ID from your store.  
The cells here will auto-populate them from the list results above.

### R-Series — single Item

In [ ]:
rsr_item = None

if lsr:
    # Use the first item from the list above, or override with a specific ID:
    #   RSR_ITEM_ID = "1234"
    RSR_ITEM_ID = rsr_items[0]["itemID"] if rsr_items else None

    if RSR_ITEM_ID:
        rsr_item = lsr.Items.get(RSR_ITEM_ID)

        print(f"Item ID : {rsr_item.itemID}")
        print(f"Name    : {rsr_item.description}")
        print(f"SKU     : {rsr_item.get('customSku', '—')}")
        print()

        # Prices are nested — nested_json_to_attr() populates the convenience fields
        rsr_item.nested_json_to_attr()
        print(f"Default price : {rsr_item.price_default}")
        print(f"MSRP          : {rsr_item.price_msrp}")
        print(f"Online price  : {rsr_item.price_online}")
        print()

        # Show all top-level keys returned by the API
        print("All fields returned:")
        pprint.pprint({k: v for k, v in rsr_item.items() if not k.startswith("_")})
    else:
        print("No items found to inspect.")
else:
    print("R-Series not configured — skipping.")

### C-Series — single Product

In [ ]:
lsc_product = None

if lsc:
    # Use the first product from the list above, or override:
    #   LSC_PRODUCT_ID = 58526124
    LSC_PRODUCT_ID = lsc_products[0]["id"] if lsc_products else None

    if LSC_PRODUCT_ID:
        lsc_product = lsc.Products.get(LSC_PRODUCT_ID)

        print(f"Product ID  : {lsc_product.id}")
        print(f"Title       : {lsc_product.title}")
        print(f"SKU         : {lsc_product.get('sku', '—')}")
        print(f"Price       : {lsc_product.get('price', '—')}")
        print(f"Visible     : {lsc_product.get('isVisible', '—')}")
        print()

        print("All fields returned:")
        pprint.pprint({k: v for k, v in lsc_product.items() if not k.startswith("_")})
    else:
        print("No products found to inspect.")
else:
    print("C-Series not configured — skipping.")

### X-Series — single Product

In [ ]:
lsx_product = None

if lsx:
    # Use the first product from the list above, or override:
    #   LSX_PRODUCT_ID = "abc-123"
    LSX_PRODUCT_ID = lsx_products[0]["id"] if lsx_products else None

    if LSX_PRODUCT_ID:
        lsx_product = lsx.Products.get(LSX_PRODUCT_ID)

        print(f"Product ID   : {lsx_product.id}")
        print(f"Name         : {lsx_product.get('name', '—')}")
        print(f"SKU          : {lsx_product.get('sku', '—')}")
        print(f"Base price   : {lsx_product.get('base_price', '—')}")
        print(f"Retail price : {lsx_product.get('retail_price', '—')}")
        print()

        print("All fields returned:")
        pprint.pprint({k: v for k, v in lsx_product.items() if not k.startswith("_")})
    else:
        print("No products found to inspect.")
else:
    print("X-Series not configured — skipping.")

## 5. Update Item Name

`item.update(**fields)` sends a PUT request with the changed fields and returns the updated object.

> ⚠️ **These cells write to your live store.**  
> Check the item ID and new value before running.

- R-Series name field: `description`  
- C-Series name field: `title`  
- X-Series name field: `name`

### R-Series — update Item name

In [ ]:
if lsr and rsr_item:
    NEW_NAME = rsr_item.description + " (updated)"  # ← change this to whatever you want

    print(f"Before: {rsr_item.description!r}")

    updated = rsr_item.update(description=NEW_NAME)

    print(f"After : {updated.description!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("R-Series not configured or no item loaded — skipping.")

### C-Series — update Product title

In [ ]:
if lsc and lsc_product:
    NEW_TITLE = lsc_product.title + " (updated)"  # ← change this

    print(f"Before: {lsc_product.title!r}")

    updated = lsc_product.update(title=NEW_TITLE)

    print(f"After : {updated.title!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("C-Series not configured or no product loaded — skipping.")

### X-Series — update Product name

In [ ]:
if lsx and lsx_product:
    NEW_NAME_X = lsx_product.get("name", "") + " (updated)"  # ← change this

    print(f"Before: {lsx_product.get('name')!r}")

    updated = lsx_product.update(name=NEW_NAME_X)

    print(f"After : {updated.get('name')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("X-Series not configured or no product loaded — skipping.")

## 6. Update Item Price

> ⚠️ **These cells write to your live store.**

Price handling differs per series:

- **R-Series** — prices live in a nested `Prices → ItemPrice` structure.  
  You update them via `item.update(Prices={"ItemPrice": [{"useTypeID": "1", "amount": "29.99"}, ...]})`.  
  `useTypeID` 1 = Default, 2 = MSRP, 3 = Online, 4 = Promotion.

- **C-Series** — `price` is a direct field on the product object.

- **X-Series** — `base_price` and `retail_price` are direct fields on the product object.

### R-Series — update Item price

In [ ]:
if lsr and rsr_item:
    # Re-fetch to get the current price structure before editing
    rsr_item = lsr.Items.get(rsr_item.itemID)
    rsr_item.nested_json_to_attr()

    NEW_DEFAULT_PRICE = "19.99"   # ← set your new price
    NEW_MSRP          = "24.99"   # ← set your new MSRP

    print(f"Before — Default: {rsr_item.price_default}  MSRP: {rsr_item.price_msrp}")

    # R-Series requires you to send back ALL price tiers, not just the one you want to change.
    # Pull the existing tiers and patch the ones you want.
    item_prices = rsr_item["Prices"]["ItemPrice"]   # list of price dicts
    item_prices[0]["amount"] = NEW_DEFAULT_PRICE    # useTypeID 1: Default
    item_prices[1]["amount"] = NEW_MSRP             # useTypeID 2: MSRP

    updated = rsr_item.update(Prices={"ItemPrice": item_prices})
    updated.nested_json_to_attr()

    print(f"After  — Default: {updated.price_default}  MSRP: {updated.price_msrp}")
    print()
    print("Full updated Prices block:")
    pprint.pprint(updated["Prices"])
else:
    print("R-Series not configured or no item loaded — skipping.")

### C-Series — update Product price

In [ ]:
if lsc and lsc_product:
    NEW_PRICE = 19.99   # ← set your new price (float, not string)

    print(f"Before: {lsc_product.get('price', '—')!r}")

    updated = lsc_product.update(price=NEW_PRICE)

    print(f"After : {updated.get('price', '—')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("C-Series not configured or no product loaded — skipping.")

### X-Series — update Product price

In [ ]:
if lsx and lsx_product:
    NEW_BASE_PRICE   = 19.99   # ← set your new base price
    NEW_RETAIL_PRICE = 24.99   # ← set your new retail price

    print(f"Before — base: {lsx_product.get('base_price')!r}  "
          f"retail: {lsx_product.get('retail_price')!r}")

    updated = lsx_product.update(
        base_price=NEW_BASE_PRICE,
        retail_price=NEW_RETAIL_PRICE,
    )

    print(f"After  — base: {updated.get('base_price')!r}  "
          f"retail: {updated.get('retail_price')!r}")
    print()
    print("Full updated object:")
    pprint.pprint({k: v for k, v in updated.items() if not k.startswith("_")})
else:
    print("X-Series not configured or no product loaded — skipping.")

---

## Tips & further exploration

### Listing all resources (pagination handled automatically)

```python
all_items = lsr.Items.listall()           # R-Series — blocks until all pages fetched
all_products = lsc.Products.listall()     # C-Series
```

### Streaming large catalogues with a generator

```python
for item in lsr.Items.iter(limit=100):    # R-Series — yields one item at a time
    print(item.itemID, item.description)

for prod in lsc.Products.iterall():       # C-Series
    print(prod.id, prod.title)
```

### Filtering results

```python
# R-Series — filter by timeStamp (incremental sync)
recent = lsr.Items.page(timeStamp=">2026-01-01T00:00:00+00:00", limit=100)

# C-Series — filter by updated date
recent = lsc.Products.page(updated_at_min="2026-01-01 00:00:00", limit=50)
```

### Counting records (C-Series only)

```python
total = lsc.Products.count()
print(f"Total products: {total}")
```

### Accessing the raw JSON

Every object has a `json` key with the raw dict that came from the API:

```python
print(rsr_item["json"])
print(lsc_product["json"])
```

### Rate limits

The connection object tracks rate-limit state automatically.  
You can inspect the last response for debugging:

```python
print(lsr.connection._last_response.headers)
```